# Assistente RAG sobre a LGPD

Notebook executável do início ao fim. Cada seção corresponde a um módulo em `src/`.

## 1. Ingestão e chunking

In [ ]:
# 1.1 Obter o repositório
REPO_URL = 'https://github.com/lucasnk1/RAG-LGPD.git'
!git clone $REPO_URL RAG-LGPD
%cd RAG-LGPD
%pip install -q pypdf chromadb sentence-transformers

### 1.2 Texto, artigos e chunks
Fonte: PDF compilado da Lei 13.709/2018 (Planalto, congelado em `data/`).

In [ ]:
from src.ingestion import *
from src.ingestion import _vetado

todos = extrair_artigos(limpar_texto(carregar_texto()))
artigos = [a for a in todos if not _vetado(a)]
print(len(todos), 'artigos no texto;', len(artigos), 'com conteúdo')
print('Excluídos (vetados/revogados):', [a.artigo for a in todos if _vetado(a)])

In [ ]:
chunks_a = chunk_por_artigo(artigos)
chunks_b = chunk_por_paragrafo(artigos)
print('Config A (artigo):', len(chunks_a), '| Config B (parágrafo):', len(chunks_b))
print(chunks_b[20])

## 2. Embeddings e ChromaDB
Modelo `intfloat/multilingual-e5-small` (prefixos `query:` / `passage:`), vetores normalizados, distância de cosseno.

In [ ]:
from pathlib import Path
from src.embeddings import MODELOS, Embedder
from src.indexing import carregar_chunks, indexar, nome_colecao

embedders = {m: Embedder(n) for m, n in MODELOS.items()}
for modelo, embedder in embedders.items():
    for chunking in ('artigo', 'paragrafo'):
        chunks = carregar_chunks(Path(f'data/processed/chunks_{chunking}.jsonl'))
        print(nome_colecao(chunking, modelo), indexar(chunks, nome_colecao(chunking, modelo), embedder).count(), 'chunks')
embedder = embedders['small']

### 2.1 Busca top-k e filtro por metadados

In [ ]:
from src.query import buscar

for r in buscar('Qual o prazo para comunicar um incidente de segurança?', 'lgpd_artigo_small', embedder, k=5):
    print(r['artigo'], round(r['distancia'], 3))

print([(r['artigo'], r['capitulo']) for r in buscar('sanções aplicáveis pela ANPD', 'lgpd_artigo_small', embedder, k=3, where={'capitulo': 'VIII'})])

## 3. Avaliação da recuperação
4 configurações: {e5-small, e5-base} x {chunk por artigo, chunk por parágrafo}.

In [ ]:
!python -m src.evaluation